In [19]:
import numpy as np
from scipy.integrate import nquad
from numpy.polynomial.legendre import leggauss

\begin{equation}
    f1 = \int_{0}^{1}\int_{-\infty}^{\infty}  e^{-y^2} x^2 dydx= \frac{\sqrt{\pi}}{3} 
\end{equation}

\begin{equation}
    f2 = \int_{0}^{\pi}\int_{-\infty}^{\infty}  e^{-|y|} \sin(x) dy dx= 4
\end{equation}

In [20]:
#testing functions 

def f1(x, y):
    return np.exp(-y**2) * x**2

def f2(x, y):
    return np.exp(-np.abs(y)) * np.sin(x)    

In [21]:
# Inner integral: finite interval with Gauss-Legendre quadrature
def integrate_x(y, a, b, n=20):
    # Gauss-Legendre nodes and weights on [-1, 1]
    nodes, weights = leggauss(n)
    # Map nodes from [-1, 1] to [a, b]
    mapped_nodes = 0.5 * (nodes * (b - a) + (b + a))
    mapped_weights = 0.5 * (b - a) * weights
    # Evaluate f(x, y) at the nodes
    values = f2(mapped_nodes, y)
    return np.sum(values * mapped_weights)

# Outer integral: infinite range with nquad
def hybrid_integral(a, b):
    def integrand(y):
        return integrate_x(y, a, b, n=20)
    result, err = nquad(lambda y: integrand(y), [[-np.inf, np.inf]])
    return result, err

# Example of f1 
res, err = hybrid_integral(0, np.pi)
print("Integral result:", res, " ± ", err)


Integral result: 3.999999999999997  ±  2.3370427715721285e-10


\begin{equation}
    f3 = \int_{0}^{1}\int_{0}^{1}\int_{-\infty}^{\infty}\int_{-\infty}^{\infty}  (x+y)e^{-(v_x^2 + v_y^2)} dv_x dv_y dx dy= \pi
\end{equation}

\begin{equation}
    f4 = \int_{0}^{\pi}\int_{0}^{\pi}\int_{-\infty}^{\infty}\int_{-\infty}^{\infty}  \sin(x)\cos(y)\frac{1}{(1 + v_x^2)(1 + v_y^2)} dv_x dv_y dx dy= 0
\end{equation}

In [22]:
def f3(x,y,vx,vy):
    return (x + y) * np.exp(-(vx**2 + vy**2))


def f4(x,y,vx,vy):
    return np.sin(x) * np.cos(y) * 1/((1 + vx**2)*(1 + vy**2))

In [23]:
# Finite-range Gauss–Legendre integration over x,y
def integrate_xy(vx, vy, ax, bx, ay, by, n=20):
    # Gauss–Legendre nodes and weights
    nodes, weights = leggauss(n)

    # Map nodes from [-1, 1] to [ax, bx] and [ay, by]
    x_nodes = 0.5 * (nodes * (bx - ax) + (bx + ax))
    y_nodes = 0.5 * (nodes * (by - ay) + (by + ay))
    x_weights = 0.5 * (bx - ax) * weights
    y_weights = 0.5 * (by - ay) * weights

    # Tensor product quadrature
    total = 0.0
    for i, xi in enumerate(x_nodes):
        for j, yj in enumerate(y_nodes):
            total += f4(xi, yj, vx, vy) * x_weights[i] * y_weights[j]
    return total

# Outer integral over infinite vx, vy using nquad
def hybrid_integral(ax, bx, ay, by, n=20):
    def integrand(vx, vy):
        return integrate_xy(vx, vy, ax, bx, ay, by, n=n)

    # nquad over (-∞, ∞) × (-∞, ∞)
    result, err = nquad(integrand, [[-np.inf, np.inf], [-np.inf, np.inf]])
    return result, err

# Example usage
res, err = hybrid_integral(0, np.pi, 0, np.pi, n=20)
print("Integral result:", res, " ± ", err)

Integral result: 3.612308891222126e-15  ±  6.566233513080693e-16


In [16]:
def f5(x,y,z,vx,vy,vz):
    return x * y**2 * z**2 * np.exp(-(vx**2 + vy**2 + vz**2))

In [18]:
# Finite-range Gauss–Legendre integration over x,y
def integrate_xy(vx, vy, vz, ax, bx, ay, by, az, bz, n=20):
    # Gauss–Legendre nodes and weights
    nodes, weights = leggauss(n)

    # Map nodes from [-1, 1] to [ax, bx] and [ay, by]
    x_nodes = 0.5 * (nodes * (bx - ax) + (bx + ax))
    y_nodes = 0.5 * (nodes * (by - ay) + (by + ay))
    z_nodes = 0.5 * (nodes * (bz - az) + (bz + az))
    x_weights = 0.5 * (bx - ax) * weights
    y_weights = 0.5 * (by - ay) * weights
    z_weights = 0.5 * (bz - az) * weights

    # Tensor product quadrature
    total = 0.0
    for i, xi in enumerate(x_nodes):
        for j, yj in enumerate(y_nodes):
            for k, zk in enumerate(z_nodes):
                total += f5(xi, yj, zk, vx, vy, vz) * x_weights[i] * y_weights[j] * z_weights[k]
    return total

# Outer integral over infinite vx, vy using nquad
def hybrid_integral(ax, bx, cx, ay, by, cy, n=20):
    def integrand(vx, vy, vz):
        return integrate_xy(vx, vy, vz, ax, bx, cx, ay, by, cy, n=n)

    # nquad over (-∞, ∞) × (-∞, ∞)
    result, err = nquad(integrand, [[-np.inf, np.inf], [-np.inf, np.inf], [-np.inf, np.inf]])
    return result, err

# Example usage
res, err = hybrid_integral(0, 1, 0, np.pi, -1, 1, n=20)
print("Integral result:", res, " ± ", err)

KeyboardInterrupt: 

In [ ]:
def integrate_xy(P_xy, a, b, r, n_r=50, n_theta=100):
    rs, ws_r = np.polynomial.legendre.leggauss(n_r)
    # Transformar de [-1,1] a [0, r]
    rs = 0.5 * (rs + 1) * r
    ws_r = ws_r * 0.5 * r  # dr = r/2

    # Cuadratura gaussiana en theta de 0 a 2pi
    thetas, ws_theta = np.polynomial.legendre.leggauss(n_theta)
    thetas = 0.5 * (thetas + 1) * 2 * np.pi
    ws_theta = ws_theta * np.pi  # dtheta = pi

    # Crear mallas para r y theta
    rs_mesh, thetas_mesh = np.meshgrid(rs, thetas, indexing='ij')
    ws_r_mesh, ws_theta_mesh = np.meshgrid(ws_r, ws_theta, indexing='ij')

    # Coordenadas cartesianas
    xs = a + rs_mesh * np.cos(thetas_mesh)
    ys = b + rs_mesh * np.sin(thetas_mesh)

    # Evaluar la función en la malla
    integrando = P_xy_vectorized(xs, ys) * rs_mesh

    # Sumar todos los puntos ponderados
    integral = np.sum(integrando * ws_r_mesh * ws_theta_mesh)
    return integral

# Outer integral over infinite vx, vy using nquad
def hybrid_integral(ax, bx, ay, by, n=20):
    def integrand(vx, vy):
        return integrate_xy(vx, vy, ax, bx, ay, by, n=n)

    # nquad over (-∞, ∞) × (-∞, ∞)
    result, err = nquad(integrand, [[-np.inf, np.inf], [-np.inf, np.inf]])
    return result, err

# Example usage
res, err = hybrid_integral(0, np.pi, 0, np.pi, n=20)
print("Integral result:", res, " ± ", err)